<a href="https://colab.research.google.com/github/antonypradeep54/RAG---QA-System-using-LlamaIndex-on-Coca-Cola-Data/blob/main/C12_Project_QA_System_using_LlamaIndex_on_Coca_Cola_data_Completed_14_Jun_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project - QA System on Coca Cola data

## Tabel of Contents
* [Data loading](#DataLoading)
* [Embedding](#DataIngestion)
* [Indexing](#Indexing)
* [Response Synthesis](#ResponseSynthesis)
* [Query Engine](#QueryEngine)
* [Basic prompts](#BasicPrompts)
* [Sentence window retriever](#SentenceWindowRetriever)
* [Auto merging retriever](#AutoMergingRetriever)
* [Auto retriever](#AutoRetriever)
* [Router query engine](#RouterQueryEngine)
* [Sub question query engine](#SubQuestionQueryEngine)

### Setup

In [ ]:
%%capture
!mkdir data
!wget "https://investors.coca-colacompany.com/filings-reports/all-sec-filings/content/0000021344-25-000011/ko-20241231.htm" -O 'data/CocaCola_annual_10K_data.pdf'


In [ ]:
import os
print(os.getcwd())
os.chdir('./data')
print(os.getcwd())
os.listdir


In [ ]:
%%capture
!pip install llama-index-readers-file pymupdf

In [ ]:
from llama_index.readers.file import PyMuPDFReader
pdf_reader = PyMuPDFReader()
documents = pdf_reader.load(file_path="/content/data/CocaCola_annual_10K_data.pdf")

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=100)
nodes = splitter.get_nodes_from_documents(documents)
len(nodes)

In [ ]:
nodes[0]

In [ ]:
nodes[0].text

In [ ]:
nodes[0].metadata

##

In [ ]:
%%capture
!pip install llama_index-embeddings-huggingface

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
BGE_embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

## Indexing with VectorStoreIndex

In [ ]:
%%capture
!pip install llama-index-embeddings-openai

In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OpenAI_API_Key')

In [ ]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex(nodes, embed_model=BGE_embed_model)
index.storage_context.persist(persist_dir="./index_storage")

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir="./index_storage")

index = load_index_from_storage(storage_context, embed_model=BGE_embed_model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

persist_dir = "/content/drive/MyDrive/index_storage"
index.storage_context.persist(persist_dir=persist_dir)

In [ ]:
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
index = load_index_from_storage(storage_context, embed_model=BGE_embed_model)

In [ ]:
%%capture
!pip install llama-index-llms-openai

In [ ]:
# configure retriever
retriever = index.as_retriever()

In [ ]:
#configure response synthesizer
from llama_index.core import get_response_synthesizer

response_synthesizer = get_response_synthesizer(response_mode="compact")

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

# assuming you have an index object already
retriever = index.as_retriever()

# create the query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer
)

# now you can query
response = query_engine.query("Tell me about Coca Cola's business")
print(response)

In [ ]:
from IPython.display import Markdown, display

In [ ]:
display(Markdown(f"### Response:\n\n{response}"))

In [ ]:
# now you can query
response = query_engine.query("Tell me about Coca Cola's risk")
print(response)

In [ ]:
display(Markdown(f"### Response:\n\n{response}"))

In [ ]:
# now you can query
response = query_engine.query("Tell me about Coca Cola's Properties")
print(response)

In [ ]:
display(Markdown(f"### Response:\n\n{response}"))

In [ ]:
from IPython.display import Markdown, display

response = query_engine.query("Tell me about Coca Cola's business model")
text = response.response if hasattr(response, "response") else str(response)

display(Markdown(f"### Response:\n\n{text}"))

## Setup LLM

In [ ]:
%%capture
!pip install llama-index-llms-openai

In [ ]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o", temperature=0)

## Create different retrievers

## Sentence window retrievers

In [ ]:
from llama_index.core.node_parser import SentenceWindowNodeParser
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

In [ ]:
Sentence_Window_Index = VectorStoreIndex(nodes, embed_model=BGE_embed_model, show_progress=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

persist_dir = "/content/drive/MyDrive/index_storage"
Sentence_Window_Index.storage_context.persist(persist_dir=persist_dir)

In [ ]:
storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
Sentence_Window_Index_StorageContext = load_index_from_storage(storage_context, embed_model=BGE_embed_model)

In [ ]:
query_engine = Sentence_Window_Index_StorageContext.as_query_engine()

In [ ]:
query = "Tell me about Coca Cola's business model?"

In [ ]:
response = query_engine.query(query)

In [ ]:
Max_SentenceWindow_score = 0.0

In [ ]:
if hasattr(response, 'source_nodes'):
    print("\nSource Nodes:")
    for i, node in enumerate(response.source_nodes):
        text = node.node.text[:500]
        score = node.score if hasattr(node, 'score') else 'N/A'
        if score > Max_SentenceWindow_score:
            Max_SentenceWindow_score = score
        print(f"Node {i+1} (Score: {score:.4f}):\n{text}\n{'-'*80}\n")

In [ ]:
print("Final Answer:")
print(response)

In [ ]:
from IPython.display import Markdown, display

#response = query_engine.query("Tell me about Coca Cola's business model")
text = response.response if hasattr(response, "response") else str(response)

display(Markdown(f"### Response:\n\n{text}"))

## Automerging retrievers

In [ ]:
from llama_index.core.node_parser import HierarchicalNodeParser, SentenceSplitter

In [ ]:
Hierarchical_Node_Parser = HierarchicalNodeParser.from_defaults()

In [ ]:
AutoMerging_nodes = Hierarchical_Node_Parser.get_nodes_from_documents(documents)

In [ ]:
len(AutoMerging_nodes)

In [ ]:
from llama_index.core.node_parser import get_leaf_nodes, get_root_nodes

In [ ]:
leaf_nodes = get_leaf_nodes(AutoMerging_nodes)

In [ ]:
len(leaf_nodes)

In [ ]:
root_nodes = get_root_nodes(AutoMerging_nodes)

In [ ]:
len(root_nodes)

In [ ]:
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage import StorageContext
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

docstore = SimpleDocumentStore()

# insert nodes into docstore
docstore.add_documents(AutoMerging_nodes)

# define storage context (will include vector store by default too)
storage_context = StorageContext.from_defaults(docstore=docstore)

In [ ]:
## Load index into vector index
from llama_index.core import VectorStoreIndex

AutoMerging_index = VectorStoreIndex(
    leaf_nodes,
    embed_model = OpenAIEmbedding(model='text-embedding-3-small'),
    storage_context=storage_context,
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
persist_path = "/content/drive/MyDrive/llamaindex/AutoMerging_index"
AutoMerging_index.storage_context.persist(persist_dir=persist_path)

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.embeddings.openai import OpenAIEmbedding

persist_path = "/content/drive/MyDrive/llamaindex/AutoMerging_index"
AutoMerging_storage_context = StorageContext.from_defaults(persist_dir=persist_path)

AutoMerging_Index_StorageContext = load_index_from_storage(
    AutoMerging_storage_context,
    embed_model=OpenAIEmbedding(model='text-embedding-3-small')
)

In [ ]:
AutoMerging_query_engine = AutoMerging_Index_StorageContext.as_query_engine()

In [ ]:
AutoMerging_response = AutoMerging_query_engine.query(query)

In [ ]:
Max_AutoMerging_score = 0.0

In [ ]:
if hasattr(AutoMerging_response, 'source_nodes'):
    print("\nSource Nodes:")
    for i, node in enumerate(AutoMerging_response.source_nodes):
        text = node.node.text[:500]
        score = node.score if hasattr(node, 'score') else 'N/A'
        if score > Max_AutoMerging_score:
            Max_AutoMerging_score = score
        print(f"Node {i+1} (Score: {score:.4f}):\n{text}\n{'-'*80}\n")

In [ ]:
print(Max_AutoMerging_score)

In [ ]:
print("Final Answer from AutoMerging:")
print(AutoMerging_response)

In [ ]:
from IPython.display import Markdown, display

#response = query_engine.query("Tell me about Coca Cola's business model")
text = AutoMerging_response.response if hasattr(AutoMerging_response, "response") else str(AutoMerging_response)

display(Markdown(f"### Response from AutoMerging :\n\n{text}"))

## Auto retriever

In [ ]:
%%capture
!pip install llama-index
!pip install llama-index-vector-stores-chroma

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

In [ ]:
from llama_index.core.schema import TextNode

nodes = [
    TextNode(
        text=(
            "The Coca-Cola Company is a total beverage company, and beverage products bearing our trademarks, sold in the United States since 1886, are now sold in more than 200 countries and territories."
            "We own or license and market numerous beverage brands, which we group into the following categories: Trademark Coca-Cola; sparkling flavors; water, sports, coffee and tea; juice, value-added dairy and plant-based beverages; and emerging beverages."
            "We own and market several of the world’s largest nonalcoholic sparkling soft drink brands, including Coca-Cola, Sprite, Coca-Cola Zero Sugar, Fanta and Diet Coke/Coca-Cola Light."
            "We make our branded beverage products available to consumers throughout the world through our network of independent bottling partners, distributors, wholesalers and retailers as well as our consolidated bottling and distribution operations. "
            "Beverages bearing trademarks owned by or licensed to the Company account for 2.2 billion of the estimated 65 billion servings of all beverages consumed worldwide every day."
            "We believe our success depends on our ability to connect with consumers by providing them with a wide variety of beverage options to meet their desires, needs and lifestyles."
            "Our success further depends on the ability of our people to execute effectively, every day."
            "We are guided by our purpose, which is to refresh the world and make a difference. Our vision for growth has three connected pillars:"
            "Loved Brands. We craft meaningful brands and a choice of drinks that people love and enjoy and that refresh them in body and spirit."
            "Done Sustainably. We grow our business in ways that achieve positive change in the world and build a more sustainable future for our planet."
            "For a Better Shared Future. We invest to improve people’s lives, from our employees to all those who touch our business system, to our investors, to the communities we call home."
            "The Coca-Cola Company was incorporated in September 1919 under the laws of the State of Delaware and succeeded to the business of a Georgia corporation with the same name that had been organized in 1892."
            "widely regarded as one of the greatest basketball players of all time."
        ),
        metadata={
            "category": "Business",
            "chapter": "Part I - ITEM 1",
        },
    ),
    TextNode(
        text=(
            "Our business, operating results, financial condition and liquidity may be adversely affected by changes in global economic conditions, including global inflationary pressures, prevailing interest rates, credit market conditions, increased unemployment, levels of consumer and business confidence, bank failures, commodity (including energy) prices and supply, a recession or economic slowdown, trade policies, foreign currency exchange rates, changing policy positions or priorities, governmental rules and approaches to taxation, levels of government spending and deficits, and actual or anticipated default on sovereign debt."
            "Many of the jurisdictions in which our products are sold have experienced, and could continue to experience, unfavorable changes in economic conditions, which could negatively affect the affordability of, and consumer demand for, our beverages, and certain markets in which our products are sold experienced intensified inflation throughout 2024, which may continue in 2025."
            "Under difficult economic conditions, consumers may seek to reduce discretionary spending by forgoing purchases of our products or by shifting away from our beverages to lower-priced products offered by other companies, including private-label brands, which could reduce our profitability and negatively affect our overall financial performance."
            "In addition, the occurrence of global or regional health events, and any related governmental, private sector and individual consumer responses, could contribute to a recession, depression or global economic downturn."
            "Other financial uncertainties in our major markets and unstable geopolitical conditions or events in certain markets, including international conflicts, civil unrest, acts of war, terrorism, governmental changes, or changes in international relations, could undermine global consumer confidence and reduce consumers’ purchasing power, thereby reducing demand for our products."
            "Throughout 2024, the Company faced disruption to our operations due to international conflicts, including the conflict between Russia and Ukraine and conflicts in the Middle East."
            "Geopolitical instability has in the past led, and may in the future lead, to logistical, transportation and supply chain disruptions; business disruptions (including labor shortages); increased risk of cybersecurity incidents or other disruptions to our information systems; reduced availability and increased costs of transportation, energy, packaging, raw materials and other input costs; and heightened security risk, impacting employee safety and/or damage to infrastructure or our assets."
            "At times, we have faced product boycotts resulting from political activism, which have reduced demand for our products. Restrictions on our ability to transfer earnings or capital across borders; price controls; limitations on profits; the negotiation of new trade agreements; new, expanded or retaliatory tariffs; import authorization requirements; and other restrictions on business activities, which have been or may be imposed or expanded as a result of political and economic instability, deterioration of economic relations between countries or otherwise, could impact our profitability."
            "In addition, U.S. trade sanctions against countries designated by the U.S. government as state sponsors of terrorism and/or financial institutions accepting transactions for commerce within such countries could increase significantly, which could make it difficult, or even impossible, for us to continue to make sales to bottlers in such countries. The imposition of retaliatory sanctions against U.S. multinational corporations by countries that are or may become subject to U.S. trade sanctions, or the delisting of our branded products by retailers in various countries in reaction to U.S."
            "trade sanctions or other governmental actions or policies, could also negatively affect our business."
        ),
        metadata={
            "category": "Risk Factors",
            "chapter": "Part I - ITEM 1A",
        },
    ),
    TextNode(
        text=(
            "Our worldwide headquarters is located on a 35-acre complex in Atlanta, Georgia."
            "The complex includes several office buildings which are used by Corporate employees and North America operating segment employees."
            "In addition, the complex includes technical and engineering facilities along with a reception center."
            "We own or lease additional facilities, real estate and office space throughout the world, which we use for administrative, manufacturing, processing, packaging, storage, warehousing, distribution and retail operations."
            "These properties are generally included in the geographic operating segment in which they are located, with the exception of our Costa retail stores, which are included in the Global Ventures operating segment, and facilities related to our consolidated bottling and distribution operations, which are included in the Bottling Investments operating segment."
            "Management believes that our Company’s facilities used for the production of our products are suitable and adequate, that they are being appropriately utilized in line with past experience, and that they have sufficient production capacity for their present intended purposes."
            "The extent of utilization of our production facilities varies based upon seasonal demand for our products."
            "However, management believes that, with the exception of certain dairy products that require specialized equipment, additional production can be achieved at the existing facilities by adding personnel and capital equipment or, at some facilities, by adding shifts of personnel or expanding the facilities. The Company is in the process of increasing our dairy production capacity."
            "We continuously review our anticipated requirements for facilities and, on the basis of that review, may from time to time acquire or lease additional facilities and/or dispose of existing facilities."

        ),
        metadata={
            "category": "Properties",
            "chapter": "Part I - ITEM 2",
        },
    ),
    TextNode(
        text=(
            "During the fiscal quarter ended December 31, 2024, none of our Directors or officers (as defined in Rule 16a-1(f) under the Exchange Act) adopted or terminated a “Rule 10b5-1 trading arrangement” or a “non-Rule 10b5-1 trading arrangement,” as each term is defined in Item 408 of Regulation S-K, except as follows:"
            "Jennifer K. Mann, Executive Vice President and President, North America operating unit, adopted a Rule 10b5-1 trading arrangement on November 22, 2024 for the potential exercise of vested stock options and the associated sale of up to 80,820 shares of common stock of the Company, subject to certain conditions."
            "The arrangement’s expiration date is November 1, 2025, or such earlier date upon which all transactions are completed."
            "This trading plan was adopted during an open trading window."
        ),
        metadata={
            "category": "Other Information",
            "chapter": "Part II - ITEM 9B",
        },
    ),

]

In [ ]:
%%capture
!pip install chromadb

In [ ]:
import chromadb
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.get_or_create_collection("quickstart")


In [ ]:
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
Auto_index = VectorStoreIndex(nodes, storage_context=storage_context)

In [ ]:
Auto_query_engine = Auto_index.as_query_engine()

In [ ]:
Auto_response = Auto_query_engine.query(query)

In [ ]:
Max_Auto_score = 0.0

In [ ]:
if hasattr(Auto_response, 'source_nodes'):
    print("\nSource Nodes:")
    for i, node in enumerate(Auto_response.source_nodes):
        text = node.node.text[:500]
        score = node.score if hasattr(node, 'score') else 'N/A'
        if score > Max_Auto_score:
            Max_Auto_score = score
        print(f"Node {i+1} (Score: {score:.4f}):\n{text}\n{'-'*80}\n")

In [ ]:
print(Max_Auto_score)

In [ ]:
print("Final Answer from Auto:")
print(Auto_response)

In [ ]:
from IPython.display import Markdown, display

#response = query_engine.query("Tell me about Coca Cola's business model")
text = Auto_response.response if hasattr(Auto_response, "response") else str(Auto_response)

display(Markdown(f"### Response from Auto :\n\n{text}"))

## Retrieval evaluation conclusion

In [ ]:
print(f"""Retriever score comparison with SentenceWindowRetrieval, AutoMerging, and Auto retrieval is given below.

 Sentance Window Retriever: {Max_SentenceWindow_score}
 Auto Merging Retriever: {Max_AutoMerging_score}
 Auto Retriever: {Max_Auto_score}

 Sentance Window Retriever is showing better perfromance than Auto Merging and Auto Sentance Window Retriever.
""")

## Router Query Engine

Router query engine enables efficient quering and data retrieval across multiple sources or services. </br>

I have implemented router query engine by creating two index (vector & summary), followed by creating two query engine (vector & summary) based on the index respectively. Two tools vector and summary are defined that the router engine can choose from based on user query.

</br>

Behaviour of the router query is explained in conclusion section.




In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
%%capture
!pip install llama-index

In [ ]:
import logging
import sys

# Set up the root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # Set logger level to INFO

# Clear out any existing handlers
logger.handlers = []

# Set up the StreamHandler to output to sys.stdout (Colab's output)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)  # Set handler level to INFO

# Add the handler to the logger
logger.addHandler(handler)


In [ ]:
from llama_index.core import VectorStoreIndex, SummaryIndex, SimpleDirectoryReader, StorageContext
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from IPython.display import display, HTML

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

In [ ]:
cwd = os.getcwd()
print("Current working directory:", cwd)

In [ ]:
%%capture
!mkdir data
!wget "https://investors.coca-colacompany.com/filings-reports/all-sec-filings/content/0000021344-25-000011/ko-20241231.htm" -O 'data/CocaCola_annual_10K_data.pdf'


In [ ]:
import os
print(os.getcwd())
os.chdir('./data')
print(os.getcwd())
os.listdir

In [ ]:
Document_2025 = SimpleDirectoryReader(input_files=['/The_Coca_Cola_Company_10K_Feb_2025.pdf']).load_data()

In [ ]:
nodes_2025 = Settings.node_parser.get_nodes_from_documents(Document_2025, chunk_size=1024)

## Define Summary/Vector Index, Query Engine, and Tools

In [ ]:
llm = OpenAI(model="gpt-3.5-turbo")

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding(model="text-embedding-3-small")

summary_index_2025 = SummaryIndex(nodes_2025, embed_model=embed_model,llm=llm)

vector_index_2025 = VectorStoreIndex(nodes_2025, embed_model=embed_model)

In [ ]:
summary_query_engine_2025 = summary_index_2025.as_query_engine(response_mode="tree_summarize",use_async=True,)

vector_query_engine_2025 = vector_index_2025.as_query_engine()

In [ ]:
from llama_index.core.tools.query_engine import QueryEngineTool

# Summary Index tool
summary_tool_2025 = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine_2025,
    description="Useful for summarization of Coca Cola 10K.",
)

# Vector Index tool
vector_tool_2025 = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine_2025,
    description="Useful for retrieving specific context from Coca Cola 10K.",
)

## Define Router Query Engine

In [ ]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors.llm_selectors import LLMSingleSelector, LLMMultiSelector
from llama_index.core.selectors.pydantic_selectors import PydanticMultiSelector, PydanticSingleSelector

In [ ]:
# Create Router Query Engine
router_query_engine_2025 = RouterQueryEngine(
    selector=PydanticSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool_2025,
        vector_tool_2025,
    ],
)

In [ ]:
response = router_query_engine_2025.query("Give me summary of this documents in 10 bullet points?")

In [ ]:
print(response)

In [ ]:
display(HTML(f'<p style="font-size:14px">{response.response}</p>'))

In [ ]:
response = router_query_engine_2025.query("Tell me about Coca Cola leage issues?")

In [ ]:
display(HTML(f'<p style="font-size:14px">{response.response}</p>'))

## Conclusion of router query engine </br>

When "Give me summary of this documents in 10 bullet points?" is prompted, its routed to summary tool and the response generated is a summary with  10 bullet points. </br>

</br>
When "Tell me about Coca Cola leage issues?" is prompted, its routed to summary vector and the generate response associated with leagel issues of Coca Cola. </br>

## Sub question query engine

In [ ]:
%%capture
!pip install llama-index

In [ ]:
from llama_index.core import SimpleDirectoryReader

In [ ]:
Document_2014 = SimpleDirectoryReader(input_files=['/The_Coca_Cola_Company_10K_Feb_2014.pdf']).load_data()

In [ ]:
from llama_index.core.settings import Settings
from llama_index.core import VectorStoreIndex

In [ ]:
nodes_2014 = Settings.node_parser.get_nodes_from_documents(Document_2014, chunk_size=1024)

In [ ]:
vector_index_2014 = VectorStoreIndex(nodes_2014, embed_model=embed_model,llm=llm)

In [ ]:
vector_query_engine_2014 = vector_index_2014.as_query_engine()

In [ ]:
#Document_2014_index = VectorStoreIndex.from_documents(Simple_Documents_2014)
#query_engine_2014 = Document_2014_index.as_query_engine()

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

In [ ]:
tool_2025 = QueryEngineTool(
    query_engine=vector_query_engine_2025,
    metadata={
        "name": "Coca_Cola_2025",
        "description": "Answers questions from Coca Cola 2025"
    }
)

tool_2014 = QueryEngineTool(
    query_engine=vector_query_engine_2014,
    metadata={
        "name": "Coca_Cola_2014",
        "description": "Answers questions from Coca Cola 2014"
    }
)

In [ ]:
tools = [tool_2025, tool_2014]

In [ ]:
tools[0].metadata

In [ ]:
from llama_index.core.query_engine import SubQuestionQueryEngine

In [ ]:
sub_query_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=tools)

In [ ]:
tools = [
    QueryEngineTool(
        query_engine=vector_query_engine_2025,
        metadata=ToolMetadata(
            name="Coca_Cola_2025",
            description="Answers questions from Coca Cola 2025"
        )
    ),
    QueryEngineTool(
        query_engine=vector_query_engine_2014,
        metadata=ToolMetadata(
            name="Coca_Cola_2014",
            description="Answers questions from Coca Cola 2014"
        )
    )
]

In [ ]:
sub_query_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=tools)

In [ ]:
query = "Give comprehensive detail about Coca Cola's risk factors in 2014 and comprehensive detail about Coca Cola's risk factors in 2025?"
response = sub_query_engine.query(query)
print(response)

In [ ]:

display(Markdown(f"### Response:\n\n{response}"))

In [ ]:
if "2014" in str(response) and "2025" in str(response):
    # Basic split using years
    response_text = str(response)
    parts = response_text.split("2025")

    # Attempt to segment based on appearance of years
    part_2014 = parts[0].split("2014")[-1].strip()
    part_2025 = parts[1].strip() if len(parts) > 1 else ""

    print("🗓️  **2014 Risk Factors**\n")
    #print(part_2014 if part_2014 else "No details found.")
    display(Markdown(f"### Response:\n\n{part_2014}"))
    print("\n" + "-"*80 + "\n")
    print("🗓️  **2025 Risk Factors**\n")
    #print(part_2025 if part_2025 else "No details found.")
    display(Markdown(f"### Response:\n\n{part_2025}"))
else:
    # Fallback: Just show raw output if formatting is unreliable
    print(str(response))

In [ ]:
Legal Proceedings

In [ ]:
query = "Give comprehensive detail about Coca Cola's Legal Proceedings in 2014 and comprehensive detail about Coca Cola's Legal Proceedings in 2025?"
response = sub_query_engine.query(query)
print(response)

In [ ]:
if "2014" in str(response) and "2025" in str(response):
    # Basic split using years
    response_text = str(response)
    parts = response_text.split("2025")

    # Attempt to segment based on appearance of years
    part_2014 = parts[0].split("2014")[-1].strip()
    part_2025 = parts[1].strip() if len(parts) > 1 else ""

    print("🗓️  **2014 Risk Factors**\n")
    #print(part_2014 if part_2014 else "No details found.")
    display(Markdown(f"### Response:\n\n{part_2014}"))
    print("\n" + "-"*80 + "\n")
    print("🗓️  **2025 Risk Factors**\n")
    #print(part_2025 if part_2025 else "No details found.")
    display(Markdown(f"### Response:\n\n{part_2025}"))
else:
    # Fallback: Just show raw output if formatting is unreliable
    print(str(response))